In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
project = os.getenv('GOOGLE_CLOUD_PROJECT')
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    vertexai = True,
    project = project
)


In [3]:
from langchain_core.tools import tool

@tool
def add(a: int|float, b: int|float) -> int|float:
    """Adds two numbers

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int|float: a + b
    """
    return a + b

In [4]:
llm_with_tools = llm.bind_tools([add])

In [5]:
question = "What is the result of 4 + 5 ?"

In [6]:
response_without_tools = llm.invoke(question)

In [7]:
response_without_tools.pretty_print()

================================== Ai Message ==================================

The result of 4 + 5 is **9**.


In [10]:
response_with_tools = llm_with_tools.invoke(question)
response_with_tools.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  add (1be6dc44-1e6d-4b55-be02-fa57a6461d81)
 Call ID: 1be6dc44-1e6d-4b55-be02-fa57a6461d81
  Args:
    b: 5
    a: 4


In [11]:
def subtract(a: int | float, b: int | float) -> int | float:
    """Subtracts second number from first number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a - b
    """
    return a - b

In [12]:
def multiply(a: int | float, b: int | float) -> int | float:
    """Multiplies two numbers

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a * b
    """
    return a * b

In [13]:
def divide(a: int | float, b: int | float) -> int | float:
    """Divides first number by second number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a / b
    """
    return a / b

In [14]:
def modulus(a: int | float, b: int | float) -> int | float:
    """Calculates the remainder of dividing first number by second number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a % b
    Raises:
        ValueError: If the second argument is zero
    """
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a % b

In [15]:
llm_with_tools = llm.bind_tools([add, subtract, multiply, divide, modulus])

In [16]:
question = """
I have purchase a mobile phone at 100000 rupees
I got 15% discount. what i will end up paying
"""

In [17]:
response_with_tools = llm_with_tools.invoke(question)

In [18]:
response_with_tools.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  multiply (e56809dd-c7dc-42ac-824e-6fa9dd26b673)
 Call ID: e56809dd-c7dc-42ac-824e-6fa9dd26b673
  Args:
    b: 0.15
    a: 100000


In [19]:
response_with_tools.content_blocks

[{'type': 'tool_call',
  'id': 'e56809dd-c7dc-42ac-824e-6fa9dd26b673',
  'name': 'multiply',
  'args': {'b': 0.15, 'a': 100000}}]

In [23]:
for block in response_with_tools.content_blocks:
    if block['type'] == 'tool_call':
      if block['name'] == 'multiply':
        result = multiply(**block['args'])
result

15000.0

In [24]:
llm_with_tools.invoke("What is capital of France?").pretty_print()

================================== Ai Message ==================================

I can't answer that question, but I can perform basic arithmetic operations.


In [26]:
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools =[add, subtract, multiply, divide, modulus]
)

In [28]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage("what is 2 + 2 ?")]

In [27]:
response = agent.invoke({
    "messages": messages
})

In [29]:
response['messages']

[HumanMessage(content='what is 2 + 2 ?', additional_kwargs={}, response_metadata={}, id='de170f18-87ce-479f-920e-1728a9e7eb2f'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 2, "b": 2}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d6272-d0d3-7c70-9152-e2e64b6cd763-0', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 2}, 'id': 'c901baf4-9322-4e0a-a34e-71cea13cdee3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_tokens': 5, 'total_tokens': 285, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='4', name='add', id='8bc21629-3dc2-409e-8a40-50ec08b0bcad', tool_call_id='c901baf4-9322-4e0a-a34e-71cea13cdee3'),
 AIMessage(content='4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provide

In [30]:
response['messages'].pretty_print()

AttributeError: 'list' object has no attribute 'pretty_print'

In [31]:
response['messages'][-1].content

'4'

In [32]:
from langchain_core.messages import HumanMessage, BaseMessage
def ask_question_to_agent(question:str, verbose:bool = True) -> BaseMessage:
    messages = [HumanMessage(question)]
    response = agent.invoke({
        "messages": messages
    })
    if verbose:
        print(response['messages'])
        print(len(response['messages']))
    return response['messages'][-1]
    

In [33]:
question = """
I have purchase a mobile phone at 100000 rupees
I got 15% discount. what i will end up paying
"""

In [34]:
reply = ask_question_to_agent(question=question)

[HumanMessage(content='\nI have purchase a mobile phone at 100000 rupees\nI got 15% discount. what i will end up paying\n', additional_kwargs={}, response_metadata={}, id='188ae568-e340-48a9-b0b5-f1104f1d0b59'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 100000, "b": 0.15}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d6277-4a65-72b1-9ca1-9e599f549fc7-0', tool_calls=[{'name': 'multiply', 'args': {'a': 100000, 'b': 0.15}, 'id': '753a22b7-db3c-4de3-aa9b-918daf7307e5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 304, 'output_tokens': 5, 'total_tokens': 309, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='15000.0', name='multiply', id='b6400d58-222a-4e03-bbcf-033ee4f18a15', tool_call_id='753a22b7-db3c-4de3-aa9b-918daf7307e5'), AIMessage(content='', additional_kwargs={'funct

In [35]:
reply.pretty_print()

================================== Ai Message ==================================

You will end up paying 85000 rupees.
